# 19 · Roadmap — Hypothesis-Driven Reliability Validation (Notebooks 20–22)

*VoxIntel-R: reference-free reliability estimation for an ASR → intent (SLU) cascade.*

This notebook is **documentation only** (no code). It states the four research
hypotheses precisely and describes what the next three notebooks deliver, so the
20–22 arc reads as one coherent, publication-ready study.

## Why this arc exists

Notebook 16 introduced VoxIntel-R and reported that **intent-native uncertainty
dominates** — ASR-native uncertainty adds nothing beyond it (H1 not supported).
Notebooks 17–18 confirmed this cross-dataset (SLURP → FSC) and the frozen-split
script (`src/analysis/frozen_split_eval.py`) removed the "provisional" caveat.

A literature review against prior SLU-reliability work surfaced four gaps that a
conference reviewer would raise, and this arc closes them:

1. **Baselines.** Our intent features are essentially *max-softmax-probability*
   (MSP) — the field's **weak** baseline. We never compared against tuned strong
   models or the full modern metric suite.
2. **ASR-signal richness.** H1 used only shallow softmax summary statistics; the
   ASR-confidence literature shows learned/beam-search confidence far exceeds
   softmax summaries.
3. **Cost.** H4 used simulated severities; cost-aware deferral needs an explicit
   cost model and the caveat that *better ranking ≠ lower deployment cost*.
4. **Distribution shift / OOD.** No shift gate; the closed-world assumption was
   never stress-tested.

## The four hypotheses

| # | Hypothesis | Decision rule |
|---|---|---|
| **H1** | ASR-native uncertainty adds predictive signal **beyond** intent-native uncertainty. | Supported iff a tuned **combined (C)** model beats **intent-only (B)** with a paired-bootstrap 95% CI on ΔROC-AUC entirely **> 0**. |
| **H2** | The reliability score is **calibrated** (or can be post-hoc calibrated). | Supported iff a calibration method drives ECE / adaptive-ECE below a small target and improves Brier. |
| **H3** | **Selective prediction** (defer high-risk cases) improves accuracy on the accepted set. | Supported iff the risk-coverage curve beats random deferral and the model AURC ≈ oracle (low E-AURC). |
| **H4** | **Cost-sensitive** deferral lowers expected action cost vs uniform confidence thresholding. | Supported iff risk-threshold deferral has lower expected cost than always-execute **and** confidence-threshold deferral, with a paired-bootstrap CI excluding 0. |

## What each notebook brings

### 20 · H1 revisited — tuned multi-model bake-off + modern metrics
- **Model comparison:** Logistic Regression, Random Forest, HistGradientBoosting,
  XGBoost, LightGBM, and an MLP — each **hyperparameter-tuned** with
  `RandomizedSearchCV` (inner CV on the training split only; the frozen 30% test
  set is never touched during selection).
- **Feature families** A (ASR-native, 9) · B (intent-native, 3) · C (combined, 12).
- **Strong single-score baselines** (MSP / entropy / margin) reported alongside.
- **Metric suite:** ROC-AUC, PR-AUC, Brier, **FPR@95%TPR**, **E-AURC**, **NCE** —
  the metrics prior work is reported in, so numbers are comparable.
- **Output:** a ranked comparison table, the H1 paired-bootstrap verdict, and the
  single best risk model persisted for 21–22. *Best result in one run.*

### 21 · H2 calibration + H3 selective prediction
- **Calibration:** uncalibrated vs Platt (sigmoid) vs isotonic vs temperature;
  ECE, adaptive-ECE, MCE, Brier + reliability diagrams.
- **Selective prediction:** risk-coverage curves and AURC / E-AURC for model-risk
  vs raw-confidence (MSP) vs oracle vs random; accuracy and coverage at target
  risk budgets (1% / 5% / 10%).

### 22 · H4 cost-sensitive decisions + shift/OOD + synthesis
- **Cost model:** an explicit `c_FN : c_FP : c_review` matrix (EcoTrust-style),
  expected-cost-vs-coverage curves, the cost-optimal deferral rule, breakeven
  analysis, and a `c_FN` sensitivity sweep.
- **Distribution shift:** SLURP → FSC transfer on the shared feature space, plus a
  Mahalanobis support gate flagging out-of-support inputs.
- **Synthesis:** one final H1–H4 verdict table.

## Shared infrastructure & reproducibility

- **Metrics/loaders:** `src/analysis/reliability_metrics.py` (self-checked via
  `python -m` `demo()`), imported by all three notebooks.
- **Frozen split:** stratified 70/30, `seed=42` — identical to
  `src/analysis/frozen_split_eval.py`, so 16/20/21/22 are directly comparable.
- **Inputs (cached, no audio / GPU / re-decode):**
  `reports/phase6_voxintel_r_slurp/voxintel_r_features.csv` (SLURP, 8,688) and
  `reports/phase7_reliability_cross_dataset/fsc_voxintel_r_features.csv` (FSC).
- **Outputs:** everything lands in `reports/phase8_hypothesis_validation/`; the
  best model in `models/voxintel_r_best_risk.joblib`.
- **Leakage guard:** the reference transcript / ground-truth columns are used only
  to build the `intent_failed` target and are asserted out of every feature set.

**Honesty note.** Everything in 20–22 runs on cached features. Two extensions need
a decode pass and are specified (not faked) in `reports/PAPER_APPENDICES.md`:
learned N-best/LM ASR confidence (Appendix A) and human severity labels (Appendix B).

## Conclusion

This notebook is the roadmap and orientation for the phase-8 hypothesis suite — documentation only, with no experiments or results of its own. It fixes the four hypotheses (H1–H4), the decision rule for each, and the shared frozen-split infrastructure so notebooks 20–22 read as one coherent, publication-ready study. The experimental payoff lives downstream: **NB20** settles H1 with a tuned multi-model bake-off, **NB21** covers H2 (calibration) and H3 (selective prediction), and **NB22** closes H4 (cost-sensitive deferral) alongside distribution-shift/OOD and the final H1–H4 synthesis. The two extensions that need a decode pass — learned ASR confidence and human severity labels — are specified, not faked, in `reports/PAPER_APPENDICES.md` (Appendices A and B).